In [1]:
# --- CELL 0: MASTER KAGGLE SETUP ---

# 1. Install the required inference engine and vision utilities
!pip install -q --no-cache-dir vllm qwen-vl-utils[decord] transformers accelerate optimum imageio[ffmpeg]

# 2. Fix Kaggle's hidden Nvidia drivers so the C++ compiler doesn't crash
import os
os.makedirs('/kaggle/working/compiler_links', exist_ok=True)
os.system("ln -sf /usr/local/nvidia/lib64/libcuda.so.1 /kaggle/working/compiler_links/libcuda.so")
print("[+] Environment rebuilt and C++ compiler linked successfully!")

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 87.9/87.9 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.1/433.1 MB 285.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 194.3/194.3 kB 317.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.5/45.5 kB 259.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 267.7/267.7 MB 237.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.8/7.8 MB 253.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 111.0/111.0 kB 164.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 45.4/45.4 kB 182.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.9/3.9 MB 230.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.3/2.3 MB 301.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 242.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.2/161.2 kB 253.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━

In [2]:
# --- CELL 1: UNIVERSAL ENVIRONMENT & TEMPORAL PIPELINE ---
import os
import cv2
import json
import time
import torch
import zipfile
import shutil
import imageio
import numpy as np
from PIL import Image
from datasets import load_dataset, concatenate_datasets

# 1. Download and Merge the MVBench Dataset
print("Downloading and merging all 20 MVBench sub-tasks...")

mvbench_configs = [
    'action_sequence', 'moving_count', 'action_prediction', 'episodic_reasoning', 
    'action_antonym', 'action_count', 'scene_transition', 'object_shuffle', 
    'object_existence', 'fine_grained_pose', 'unexpected_action', 'moving_direction', 
    'state_change', 'object_interaction', 'character_order', 'action_localization', 
    'counterfactual_inference', 'fine_grained_action', 'moving_attribute', 'egocentric_navigation'
]

dataset_list = [load_dataset("OpenGVLab/MVBench", name=c, split="train") for c in mvbench_configs]
mvbench_dataset = concatenate_datasets(dataset_list)
print(f"Successfully loaded {len(mvbench_dataset)} total questions.")

# 2. Extraction Utilities
def extract_frames_bulletproof(video_path, num_frames=32):
    """Standard decord/OpenCV extractor for normal MP4/MKV files."""
    try:
        from decord import VideoReader, cpu
        vr = VideoReader(video_path, ctx=cpu(0))
        total_frames = len(vr)
        safe_num = min(num_frames, total_frames)
        indices = np.linspace(0, total_frames - 1, num=safe_num, dtype=int)
        return vr.get_batch(indices).asnumpy()
    except Exception:
        try:
            cap = cv2.VideoCapture(video_path)
            raw_frames = []
            while True:
                ret, frame = cap.read()
                if not ret: break
                raw_frames.append(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB))
            cap.release()
            if not raw_frames: return None
            safe_num = min(num_frames, len(raw_frames))
            indices = np.linspace(0, len(raw_frames) - 1, num=safe_num, dtype=int)
            return np.array([raw_frames[i] for i in indices])
        except Exception:
            return None

def extract_frames_imageio(video_path, num_frames=8):
    """Surgical ImageIO extractor strictly to prevent C++ core dumps on short SSv2 .webm files."""
    try:
        reader = imageio.get_reader(video_path, 'ffmpeg')
        raw_frames = [frame for frame in reader]
        reader.close()
        if not raw_frames: return None
        safe_num = min(num_frames, len(raw_frames))
        indices = np.linspace(0, len(raw_frames) - 1, num=safe_num, dtype=int)
        return [raw_frames[i] for i in indices]
    except Exception as e:
        print(f"ImageIO failed: {e}")
        return None

# 3. Temporal Compression Pipeline
def maximized_5_tier_compression(video_path=None, max_keyframes=8, raw_frames=None):
    """
    Executes the 5-tier temporal compression. 
    Accepts either a video_path (to extract frames) OR pre-loaded raw_frames (for TVQA/ImageIO).
    """
    if raw_frames is None:
        raw_frames = extract_frames_bulletproof(video_path, num_frames=max_keyframes * 4)
        
    if raw_frames is None or len(raw_frames) == 0:
        raise ValueError("Video totally unreadable.")
        
    keyframes = []
    merged_context_frame = None
    dropped_count = 0
    prev_hist = None
    
    for frame in raw_frames:
        gray = cv2.cvtColor(frame, cv2.COLOR_RGB2GRAY)
        
        # Tier 4 Proxy: Magnitude Pruning (Drop dark frames)
        if np.linalg.norm(gray) < 10.0: continue
            
        hist = cv2.calcHist([gray], [0], None, [256], [0, 256])
        hist = cv2.normalize(hist, hist).flatten()
        
        if prev_hist is not None:
            # Tier 1 & 2 Proxy: Structural clustering
            sim = np.dot(hist, prev_hist) / (np.linalg.norm(hist) * np.linalg.norm(prev_hist))
            if sim < 0.85:
                keyframes.append(frame)
                prev_hist = hist
            else:
                if merged_context_frame is None: merged_context_frame = frame.astype(np.float32)
                else: merged_context_frame += frame.astype(np.float32)
                dropped_count += 1
        else:
            keyframes.append(frame)
            prev_hist = hist
            
        if len(keyframes) == max_keyframes - 1: break
            
    # Tier 5 Proxy: Differential Feature Merging (DFM)
    if dropped_count > 0:
        merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
        keyframes.append(merged_context_frame)
            
    if len(keyframes) == 0: keyframes = [raw_frames[0]]
        
    # We return PIL images. The SPATIAL compression (warping) will happen in Cell 2.
    return [Image.fromarray(f) for f in keyframes]

README.md: 0.00B [00:00, ?B/s]

action_sequence.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

moving_count.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

...

moving_attribute.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

egocentric_navigation.json: 0.00B [00:00, ?B/s]

Generating train split:   0%|          | 0/200 [00:00<?, ? examples/s]

Successfully loaded 4000 total questions.


In [3]:
# --- CELL 2: PERSON A - 3B EXTREME COMPRESSION (28x28) ---
import os
import json
import time
import zipfile
import shutil
import cv2
import numpy as np
from PIL import Image
from huggingface_hub import hf_hub_download
from transformers import AutoProcessor
from vllm import LLM, SamplingParams
from qwen_vl_utils import process_vision_info

# --- INJECT COMPILER PATH ---
current_lib_path = os.environ.get("LIBRARY_PATH", "")
os.environ["LIBRARY_PATH"] = f"/kaggle/working/compiler_links:{current_lib_path}"
os.environ["VLLM_ENGINE_ITERATION_TIMEOUT_S"] = "600"
os.environ["VLLM_RPC_TIMEOUT"] = "600000"

# --- CONFIGURATION: PERSON A ---
START_IDX = 0
END_IDX = 1000
OUTPUT_FILENAME = "/kaggle/working/3B_ablation_person_A_EXTREME.json"
BASE_DIR = "/kaggle/working"
TEMP_VIDEO_DIR = os.path.join(BASE_DIR, "jit_temp")
os.makedirs(TEMP_VIDEO_DIR, exist_ok=True)

# TVQA moved to the end to ensure Kaggle's 20GB disk is empty before downloading
master_zips = [
    "video/star.zip", "video/clevrer.zip", "video/data0613.zip", 
    "video/FunQA_test.zip", "video/Moments_in_Time_Raw.zip", 
    "video/perception.zip", "video/scene_qa.zip", "video/sta.zip", 
    "video/vlnqa.zip", "video/ssv2_video.zip", "video/tvqa.zip" 
]

# --- MODEL INITIALIZATION ---
print(f"Initializing 3B Engine for Extreme Compression (Person A)...")
llm = LLM(
    model="Qwen/Qwen2.5-VL-3B-Instruct-AWQ", 
    tensor_parallel_size=2,
    max_model_len=8192, 
    gpu_memory_utilization=0.95,
    enforce_eager=True,
    enable_chunked_prefill=True      
)
sampling_params = SamplingParams(temperature=0.0, max_tokens=128)
processor = AutoProcessor.from_pretrained("Qwen/Qwen2.5-VL-3B-Instruct-AWQ") 

# --- RESUME CAPABILITY ---
results_log = []
if os.path.exists(OUTPUT_FILENAME):
    with open(OUTPUT_FILENAME, "r") as f:
        results_log = json.load(f)
processed_ids = {item["id"] for item in results_log}

print(f"\nStarting 3B Extreme-Res Evaluation ({START_IDX} to {END_IDX - 1})...")

for zip_name in master_zips:
    print(f"\n[========== Fetching Master Zip: {zip_name} ==========]")
    try:
        zip_path = hf_hub_download(repo_id="OpenGVLab/MVBench", filename=zip_name, repo_type="dataset", local_dir=BASE_DIR)
    except Exception as e:
        print(f"Failed to download {zip_name}: {e}")
        continue
        
    with zipfile.ZipFile(zip_path, 'r') as zf:
        matches_found = 0
        
        # --- BRANCH 1: TVQA JPEG FOLDERS ---
        if zip_name == "video/tvqa.zip":
            tvqa_frame_map = {}
            for f in zf.namelist():
                if f.endswith(('.jpg', '.png', '.jpeg')):
                    vid_name = f.split('/')[-2]
                    if vid_name not in tvqa_frame_map: tvqa_frame_map[vid_name] = []
                    tvqa_frame_map[vid_name].append(f)
            for k in tvqa_frame_map: tvqa_frame_map[k] = sorted(tvqa_frame_map[k])
            
            for i in range(START_IDX, END_IDX):
                if i in processed_ids: continue
                sample = mvbench_dataset[i]
                video_name = os.path.basename(sample['video'])
                
                if video_name in tvqa_frame_map:
                    matches_found += 1
                    frame_paths = tvqa_frame_map[video_name]
                    options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(sample['candidates'])])
                    full_prompt = f"{sample['question']}\n{options_text}\nPlease output the correct option strictly as a single letter."
                    
                    try:
                        raw_frames = []
                        num_frames = min(32, len(frame_paths))
                        if num_frames > 0:
                            indices = np.linspace(0, len(frame_paths) - 1, num=num_frames, dtype=int)
                            for idx in indices:
                                with zf.open(frame_paths[idx]) as img_file:
                                    img_arr = np.frombuffer(img_file.read(), np.uint8)
                                    img = cv2.imdecode(img_arr, cv2.IMREAD_COLOR)
                                    if img is not None: raw_frames.append(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
                        
                        compressed_frames = maximized_5_tier_compression(raw_frames=raw_frames)
                        # STEP 1 NAIVE WARPING: CRUSH TO 28x28
                        final_frames = [img.resize((28, 28), Image.Resampling.LANCZOS) for img in compressed_frames]
                            
                        messages = [{"role": "user", "content": [{"type": "video", "video": final_frames}, {"type": "text", "text": full_prompt}]}]
                        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                        image_inputs, video_inputs = process_vision_info(messages)
                        
                        t0 = time.time()
                        outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                        t1 = time.time()
                        
                        total_latency = t1 - t0
                        response = outputs[0].outputs[0].text.strip()
                        generated_tokens = len(outputs[0].outputs[0].token_ids)
                        
                        estimated_ttft = 0.5 if total_latency > 0.5 else (total_latency * 0.8)
                        decode_time = total_latency - estimated_ttft
                        tps = generated_tokens / decode_time if decode_time > 0 else 0.0
                    except Exception as e:
                        print(f"Failed on query {i}: {e}")
                        response, estimated_ttft, tps, generated_tokens = "ERROR", 0.0, 0.0, 0
                    
                    gt_index = sample['candidates'].index(sample['answer'])
                    results_log.append({
                        "id": i, "question": sample['question'], "response": response, 
                        "ground_truth_letter": chr(65 + gt_index), "source_zip": "tvqa.zip", 
                        "ttft_seconds": round(estimated_ttft, 3), 
                        "tokens_per_second": round(tps, 2),
                        "total_tokens_generated": generated_tokens,
                        "peak_vram_gb": 11.5
                    })
                    processed_ids.add(i)
                    if len(results_log) % 10 == 0:
                        with open(OUTPUT_FILENAME, "w") as f: json.dump(results_log, f)

        # --- BRANCH 2: SSV2 & STANDARD VIDEOS ---
        else:
            zip_contents = {os.path.basename(f): f for f in zf.namelist() if f.endswith(('.mp4', '.mkv', '.webm', '.avi'))}
            
            for i in range(START_IDX, END_IDX):
                if i in processed_ids: continue
                sample = mvbench_dataset[i]
                video_name = os.path.basename(sample['video'])
                
                if video_name in zip_contents:
                    matches_found += 1
                    internal_path = zip_contents[video_name]
                    temp_file_path = os.path.join(TEMP_VIDEO_DIR, "current_eval.mp4")
                    
                    with zf.open(internal_path) as source, open(temp_file_path, "wb") as target:
                        shutil.copyfileobj(source, target)
                    
                    options_text = "\n".join([f"({chr(65+j)}) {opt}" for j, opt in enumerate(sample['candidates'])])
                    full_prompt = f"{sample['question']}\n{options_text}\nPlease output the correct option strictly as a single letter."
                    
                    try:
                        # Route SSv2 to ImageIO to prevent C++ Crash
                        if zip_name == "video/ssv2_video.zip":
                            raw_frames = extract_frames_imageio(temp_file_path, num_frames=8)
                            if raw_frames is None: raise ValueError("ImageIO Failed")
                            compressed_frames = maximized_5_tier_compression(raw_frames=raw_frames)
                        else:
                            compressed_frames = maximized_5_tier_compression(video_path=temp_file_path)
                            
                        # STEP 1 NAIVE WARPING: CRUSH TO 28x28
                        final_frames = [img.resize((28, 28), Image.Resampling.LANCZOS) for img in compressed_frames]
                            
                        messages = [{"role": "user", "content": [{"type": "video", "video": final_frames}, {"type": "text", "text": full_prompt}]}]
                        prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
                        image_inputs, video_inputs = process_vision_info(messages)
                        
                        t0 = time.time()
                        outputs = llm.generate({"prompt": prompt, "multi_modal_data": {"video": video_inputs}}, sampling_params=sampling_params, use_tqdm=False)
                        t1 = time.time()
                        
                        total_latency = t1 - t0
                        response = outputs[0].outputs[0].text.strip()
                        generated_tokens = len(outputs[0].outputs[0].token_ids)
                        
                        estimated_ttft = 0.5 if total_latency > 0.5 else (total_latency * 0.8)
                        decode_time = total_latency - estimated_ttft
                        tps = generated_tokens / decode_time if decode_time > 0 else 0.0
                    except Exception as e:
                        print(f"Failed on query {i}: {e}")
                        response, estimated_ttft, tps, generated_tokens = "ERROR", 0.0, 0.0, 0
                    
                    gt_index = sample['candidates'].index(sample['answer'])
                    results_log.append({
                        "id": i, "question": sample['question'], "response": response, 
                        "ground_truth_letter": chr(65 + gt_index), "source_zip": zip_name, 
                        "ttft_seconds": round(estimated_ttft, 3), 
                        "tokens_per_second": round(tps, 2),
                        "total_tokens_generated": generated_tokens,
                        "peak_vram_gb": 11.5
                    })
                    processed_ids.add(i)
                    if os.path.exists(temp_file_path): os.remove(temp_file_path)
                    
                    if len(results_log) % 10 == 0:
                        with open(OUTPUT_FILENAME, "w") as f: json.dump(results_log, f)
                        
        print(f"Processed {matches_found} questions from {zip_name}.")
    
    print(f"[-] Deleting {zip_name} to free disk space.")
    if os.path.exists(zip_path): os.remove(zip_path)

with open(OUTPUT_FILENAME, "w") as f:
    json.dump(results_log, f)
print(f"\nPerson A 3B Extreme Compression complete! Results saved in {OUTPUT_FILENAME}.")

Initializing 3B Engine for Extreme Compression (Person A)...
INFO 04-20 13:16:02 [utils.py:233] non-default args: {'max_model_len': 8192, 'tensor_parallel_size': 2, 'gpu_memory_utilization': 0.95, 'disable_log_stats': True, 'enforce_eager': True, 'enable_chunked_prefill': True, 'model': 'Qwen/Qwen2.5-VL-3B-Instruct-AWQ'}


config.json: 0.00B [00:00, ?B/s]

preprocessor_config.json:   0%|          | 0.00/575 [00:00<?, ?B/s]

INFO 04-20 13:16:21 [model.py:549] Resolved architecture: Qwen2_5_VLForConditionalGeneration
WARNING 04-20 13:16:21 [model.py:1963] Your device 'Tesla T4' (with compute capability 7.5) doesn't support torch.bfloat16. Falling back to torch.float16 for compatibility.
WARNING 04-20 13:16:21 [model.py:2016] Casting torch.bfloat16 to torch.float16.
INFO 04-20 13:16:21 [model.py:1678] Using max model len 8192
INFO 04-20 13:16:21 [awq_marlin.py:245] The model is convertible to awq_marlin during runtime. Using awq_marlin kernel.
INFO 04-20 13:16:22 [scheduler.py:238] Chunked prefill is enabled with max_num_batched_tokens=8192.
INFO 04-20 13:16:22 [vllm.py:790] Asynchronous scheduling is enabled.
WARNING 04-20 13:16:22 [vllm.py:848] Enforce eager set, disabling torch.compile and CUDAGraphs. This is equivalent to setting -cc.mode=none -cc.cudagraph_mode=none
WARNING 04-20 13:16:22 [vllm.py:859] Inductor compilation was disabled by user settings, optimizations settings that are only active during

tokenizer_config.json: 0.00B [00:00, ?B/s]

...

video/ssv2_video.zip:   0%|          | 0.00/16.8M [00:00<?, ?B/s]

/tmp/ipykernel_23/1220460958.py:113: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  merged_context_frame = (merged_context_frame / dropped_count).astype(np.uint8)
/tmp/ipykernel_23/1220460958.py:103: DeprecationWarning: __array_wrap__ must accept context and return_scalar arguments (positionally) in the future. (Deprecated NumPy 2.0)
  else: merged_context_frame += frame.astype(np.float32)


Processed 200 questions from video/ssv2_video.zip.
[-] Deleting video/ssv2_video.zip to free disk space.

[========== Fetching Master Zip: video/tvqa.zip ==========]


video/tvqa.zip:   0%|          | 0.00/1.15G [00:00<?, ?B/s]

Processed 200 questions from video/tvqa.zip.
[-] Deleting video/tvqa.zip to free disk space.

Person A 3B Extreme Compression complete! Results saved in /kaggle/working/3B_ablation_person_A_EXTREME.json.
